# Stage 5 — Analysis: figures + auto hypothesis verdicts

Aggregates every CSV/JSONL into six figures and `results/summary.md`, where
each pre-registered hypothesis (H0–H4) gets a machine-computed verdict against
the thresholds in `configs/analysis.yaml`. **Read summary.md first, then look
at the figures — verdicts must not depend on which figure looks nicer.**

In [ ]:
import os, sys, subprocess
from pathlib import Path

IN_KAGGLE = Path("/kaggle").is_dir()
if IN_KAGGLE:
    repo = Path("mv-brainmae")
    if not repo.is_dir():
        # set your repo URL after pushing to GitHub
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/YOUR_GITHUB_USERNAME/mv-brainmae.git"],
                       check=True)
else:
    repo = Path("..")  # local: notebooks/ inside a clone
sys.path.insert(0, str(repo.resolve()))
os.chdir(repo)
print("repo:", repo.resolve())

import torch
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
from mvbrainmae.utils import load_config
from mvbrainmae.analysis import run_analysis

cfg = load_config("configs/analysis.yaml")
print(run_analysis(cfg))

In [ ]:
# ---- VERIFICATION: print the whole report ----
print(Path("results/summary.md").read_text())

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
for f in sorted(Path("figures").glob("fig_*.png")):
    fig, ax = plt.subplots(figsize=(11, 6)); ax.imshow(Image.open(f)); ax.axis("off")
    ax.set_title(f.name); plt.tight_layout(); plt.show()

In [ ]:
# bundle everything you want to keep (Kaggle output = downloadable/versioned)
import os, zipfile
from pathlib import Path
out_zip = "/kaggle/working/mv_brainmae_artifacts.zip" if IN_KAGGLE \
          else "mv_brainmae_artifacts.zip"
with zipfile.ZipFile(out_zip, "w", zipfile.ZIP_DEFLATED) as z:
    for d in ("results", "figures"):
        for root, _, files in os.walk(d):
            for fn in files:
                z.write(os.path.join(root, fn))
    for p in Path("outputs").glob("*/encoder.pt"):
        z.write(p)
    for p in Path("outputs").glob("*/pretrain_*.jsonl"):
        z.write(p)
print("artifacts zipped ->", out_zip)